In [ ]:
# 数据预处理


In [ ]:
import pandas as pd


In [ ]:
def load_data(path):
    return pd.read_csv(path)
    
def select_columns(df):
    cols = [
        "user_id",
        "video_id",
        "timestamp",
        "is_click",
    ]
    return df[cols].copy()
    
def clean_data(df):
    df = df.dropna(
        subset=["user_id", "video_id", "timestamp", "is_click"]
    )

    df = df.drop_duplicates()

    return df


def encode_ids(df):
    user_ids = df["user_id"].unique()
    item_ids = df["video_id"].unique()

    user2id = {
        uid: idx + 1
        for idx, uid in enumerate(user_ids)
    }

    item2id = {
        iid: idx + 1
        for idx, iid in enumerate(item_ids)
    }

    df["user_idx"] = df["user_id"].map(user2id)
    df["item_idx"] = df["video_id"].map(item2id)

    return df, user2id, item2id


def sort_by_time(df):
    return df.sort_values(
        ["user_idx", "timestamp"]
    ).reset_index(drop=True)


def preprocess(input_path):
    df = load_data(input_path)

    df = select_columns(df)
    df = clean_data(df)

    df, user2id, item2id = encode_ids(df)

    df = sort_by_time(df)

    return df, user2id, item2id


if __name__ == "__main__":
    df, user2id, item2id = preprocess(
        "data/raw/kuairand.csv"
    )

    df.to_parquet(
        "data/processed/interactions.parquet",
        index=False
    )

    print(df.head())
    print("num users:", len(user2id))
    print("num items:", len(item2id))